# Proton-PottsMPNN — Google Colab

Google Colab workflow for pH-sensitive protein sequence design using Proton-PottsMPNN.

This notebook:

1. Sets up a Python 3.12 environment
2. Installs Proton-PottsMPNN and its dependencies
3. Builds/configures HBPLUS
4. Accepts a user-supplied PDB structure
5. Builds the Proton-PottsMPNN design context
6. Performs pH-sensitive sequence redesign
7. Reports sequence mutations and protonation-state tokens
8. Exports the designed sequence as FASTA
9. Exports a residue-level mutation/protonation report as CSV

## Important

The current demonstration uses a single-chain 57-residue structure.

This demonstrates the Proton-PottsMPNN workflow, but it is **not an EGFR–binder complex** and therefore does not by itself establish EGFR binding or pH-dependent biological function.

Experimental validation and structural/pKa analysis are required for a functional pH-responsive protein.

## 1. Configure the Colab runtime

For the recommended workflow:

**Runtime → Change runtime type → GPU**

Then run the next cell.

In [1]:
# Check Colab runtime

import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not available.")

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


## 2. Clone the Proton-PottsMPNN repository

This notebook uses the user's GitHub fork.

Repository:

`https://github.com/selvaedwardraja/ProtonPottsMPNN`

In [2]:
# Clone the user's Proton-PottsMPNN fork

!pip -q install uv

from pathlib import Path

REPO_DIR = Path("/content/ProtonPottsMPNN")

if not REPO_DIR.exists():
    !git clone -q https://github.com/selvaedwardraja/ProtonPottsMPNN.git {REPO_DIR}

%cd /content/ProtonPottsMPNN

print("Repository:", REPO_DIR)
print("Exists:", REPO_DIR.exists())

/content/ProtonPottsMPNN
Repository: /content/ProtonPottsMPNN
Exists: True


## 3. Create the Python 3.12 environment

Proton-PottsMPNN requires Python 3.12.

The Colab system Python may be a different version, so the workflow uses a dedicated `uv` virtual environment.

In [3]:
# Create Proton-PottsMPNN Python 3.12 environment

VENV_PYTHON = "/content/ProtonPottsMPNN/.venv/bin/python"

!uv venv --clear --python 3.12 /content/ProtonPottsMPNN/.venv

print("Python environment created:")
print(VENV_PYTHON)

Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
Python environment created:
/content/ProtonPottsMPNN/.venv/bin/python


## 4. Install Proton-PottsMPNN dependencies

In [4]:
# Install Proton-PottsMPNN dependencies

!uv pip install \
    --python /content/ProtonPottsMPNN/.venv/bin/python \
    -e ./foundry \
    -r requirements-extra.txt

Resolved 210 packages in 1.06s
Prepared 1 package in 366ms
Installed 210 packages in 997ms
 + absl-py==2.5.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.15.1
 + argon2-cffi==25.1.0
 + argon2-cffi-bindings==26.1.0
 + arrow==1.4.0
 + assertpy==1.1
 + asttokens==3.0.2
 + async-lru==2.3.0
 + atomworks==2.2.1
 + attrs==26.1.0
 + babel==2.18.0
 + beartype==0.22.9
 + beautifulsoup4==4.15.0
 + biotite==1.4.0
 + biotraj==1.2.2
 + bleach==6.4.0
 + certifi==2026.7.22
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + click==8.5.0
 + cloudpickle==3.1.2
 + comm==0.2.3
 + contourpy==1.4.0
 + cuda-bindings==13.4.3
 + cuda-pathfinder==1.8.3
 + cuda-toolkit==13.0.3.0
 + cycler==0.12.1
 + cython==3.3.0
 + cytoolz==1.1.0
 + debugpy==1.8.22
 + decorator==5.3.1
 + defusedxml==0.7.1
 + dm-tree==0.1.10
 + einops==0.8.2
 + einx==0.4.3
 + environs==11.2.1
 + executing==2.2.1
 + fastjsonschema==2.22.

In [5]:
# Verify the Proton-PottsMPNN environment

import subprocess

PYTHON = "/content/ProtonPottsMPNN/.venv/bin/python"

check_script = """
import sys
import torch
import mpnn

print("Python:", sys.version)
print("Executable:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("mpnn location:", mpnn.__file__)
"""

result = subprocess.run(
    [PYTHON, "-c", check_script],
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Proton-PottsMPNN environment verification failed.")

Python: 3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]
Executable: /content/ProtonPottsMPNN/.venv/bin/python
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
mpnn location: /content/ProtonPottsMPNN/foundry/models/mpnn/src/mpnn/__init__.py



## 5. Configure HBPLUS

Proton-PottsMPNN's structural preprocessing uses HBPLUS for hydrogen-bond annotation.

The Colab environment does not contain the developer's original local HBPLUS path, so we build HBPLUS inside the Colab runtime and explicitly set `HBPLUS_PATH`.

In [6]:
# Install and build HBPLUS

import os
import subprocess
from pathlib import Path

HBPLUS_ROOT = Path("/content/hbplus")
HBPLUS_ARCHIVE = Path("/content/hbplus.tar.gz")

if not HBPLUS_ARCHIVE.exists():

    subprocess.run(
        [
            "wget",
            "-q",
            "https://github.com/RomanLas/HBPLUS/raw/refs/heads/main/hbplus.tar.gz",
            "-O",
            str(HBPLUS_ARCHIVE),
        ],
        check=True,
    )

if not HBPLUS_ROOT.exists():

    HBPLUS_ROOT.mkdir()

    subprocess.run(
        [
            "tar",
            "-xzf",
            str(HBPLUS_ARCHIVE),
            "-C",
            str(HBPLUS_ROOT),
        ],
        check=True,
    )

makefiles = list(HBPLUS_ROOT.rglob("Makefile"))

if not makefiles:
    raise FileNotFoundError("HBPLUS Makefile not found.")

build_dir = makefiles[0].parent

subprocess.run(
    ["make"],
    cwd=build_dir,
    check=True,
)

executables = [
    p for p in build_dir.rglob("hbplus")
    if p.is_file() and os.access(p, os.X_OK)
]

if not executables:
    raise FileNotFoundError(
        "HBPLUS executable was not produced."
    )

HBPLUS_PATH = str(executables[0].resolve())

os.environ["HBPLUS_PATH"] = HBPLUS_PATH

print("HBPLUS:")
print(HBPLUS_PATH)

print("Exists:", Path(HBPLUS_PATH).exists())

HBPLUS:
/content/hbplus/hbplus/hbplus
Exists: True


In [7]:
# Verify HBPLUS

result = subprocess.run(
    [HBPLUS_PATH],
    capture_output=True,
    text=True,
)

print("HBPLUS return code:", result.returncode)

if result.stdout:
    print(result.stdout[:1000])

if result.returncode != 0:
    raise RuntimeError("HBPLUS verification failed.")

HBPLUS return code: 0
HBPLUS Hydrogen Bond Calculator v 3.2            Oct 03 13:16:59 UTC 2026
(c) I McDonald, D Naylor, D Jones and J Thornton 1993 All Rights Reserved.

Configured for 100000 atoms and 24000 residues.

Criteria

Minimum Angles; DHA 90.00, HAAA 90.00, DAAA 90.00
Maximum Distances; D-A 3.9, H-A 2.5, S-S 3.0
Maximum angles at aromatic acceptors DAAX 20.00, HAAAX 20.00
Minimum covalent separation 3 Covalent bonds

Enter directory name for output file (default current directory)

Enter the name of the next file to be processed:
just a file name for a Brookhaven Format PDB file or
@file name for a file holding names of Brookhavenformat files - or a blank line to end.




## 6. Upload a PDB structure

Upload the structure to be redesigned.

For a binder/target design, the PDB should contain the relevant complex.

For a single-chain test, the chain itself can be used as the design target.

In [8]:
# Upload PDB

from google.colab import files
from pathlib import Path

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No file uploaded.")

pdb_files = [
    Path(name)
    for name in uploaded.keys()
    if name.lower().endswith(".pdb")
]

if not pdb_files:
    raise ValueError("Please upload a PDB file.")

PDB_FILE = pdb_files[0].resolve()

print("PDB:")
print(PDB_FILE)

print("Exists:", PDB_FILE.exists())
print(
    "Size:",
    round(PDB_FILE.stat().st_size / 1024, 1),
    "KB"
)

Saving SKB_EGFR_01.pdb to SKB_EGFR_01.pdb
PDB:
/content/ProtonPottsMPNN/SKB_EGFR_01.pdb
Exists: True
Size: 35.4 KB


In [9]:
# Step 8 — Inspect PDB chains
# Use the Proton-PottsMPNN Python 3.12 environment

import os
import subprocess

PYTHON = "/content/ProtonPottsMPNN/.venv/bin/python"

script = """
from biotite.structure.io.pdb import PDBFile

pdb_file = __PDB_FILE__

atom_array = PDBFile.read(pdb_file).get_structure(model=1)

print("Total atoms:", atom_array.array_length())
print()
print("Chains:")
print("=" * 50)

for chain_id in sorted(set(atom_array.chain_id)):

    chain = atom_array[
        atom_array.chain_id == chain_id
    ]

    residue_keys = []

    for res_name, res_id in zip(
        chain.res_name,
        chain.res_id
    ):
        key = (int(res_id), res_name)

        if key not in residue_keys:
            residue_keys.append(key)

    print(
        f"Chain {chain_id}: "
        f"{len(residue_keys)} residues, "
        f"{chain.array_length()} atoms"
    )

    print(
        " First residues:",
        " ".join(
            f"{res_id}{res_name}"
            for res_id, res_name in residue_keys[:10]
        )
    )
"""

script = script.replace(
    "__PDB_FILE__",
    repr(str(PDB_FILE))
)

env = os.environ.copy()
env["HBPLUS_PATH"] = HBPLUS_PATH

result = subprocess.run(
    [PYTHON, "-c", script],
    capture_output=True,
    text=True,
    env=env,
)

print(result.stdout)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "PDB inspection failed."
    )

Total atoms: 446

Chains:
Chain A: 57 residues, 446 atoms
 First residues: 1MET 2LYS 3GLU 4ARG 5HIS 6GLU 7ILE 8LEU 9GLU 10GLU



## 7. Select the binder/design chain

For the current demonstration the uploaded structure contains a single chain:

`A`

For a different structure, change the value below to the chain that should be redesigned.

In [10]:
# Select the chain to redesign

BINDER_CHAIN = "A"

print("Design chain:", BINDER_CHAIN)

Design chain: A


In [11]:
# Build Proton-PottsMPNN design context

env = os.environ.copy()
env["HBPLUS_PATH"] = HBPLUS_PATH

script = """
from biotite.structure.io.pdb import PDBFile
from mpnn.inference_engines.potts_mpnn_ph import PottsMPNNPHEngine

checkpoint = (
    "/content/ProtonPottsMPNN/"
    "checkpoints/potts_v6_afdb_edge_his0.3_acid0.06/"
    "epoch-0125.ckpt"
)

pdb_file = __PDB_FILE__
binder_chain = __BINDER_CHAIN__

engine = PottsMPNNPHEngine(
    checkpoint_path=checkpoint,
    extended_vocab="v6",
    out_directory=None,
    write_fasta=False,
    write_structures=False,
)

atom_array = PDBFile.read(pdb_file).get_structure(model=1)

ctx = engine._build_context(
    atom_array,
    binder_chain
)

binder_res_ids = [
    int(ctx.token_aa.res_id[int(p)])
    for p in ctx.chA_free_idx.tolist()
]

print("==========================================")
print("PROTON-POTTSMPNN CONTEXT")
print("==========================================")

print("Design chain:", binder_chain)
print("Designable residues:", len(binder_res_ids))
print("Residues:", binder_res_ids)
"""

script = script.replace(
    "__PDB_FILE__",
    repr(str(PDB_FILE))
)

script = script.replace(
    "__BINDER_CHAIN__",
    repr(BINDER_CHAIN)
)

result = subprocess.run(
    [PYTHON, "-c", script],
    capture_output=True,
    text=True,
    env=env,
)

print(result.stdout)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "Proton-PottsMPNN context construction failed."
    )

PROTON-POTTSMPNN CONTEXT
Design chain: A
Designable residues: 57
Residues: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57]


STDERR:
Environment variable CCD_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository
Environment variable PDB_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomwor

## 8. Run pH-sensitive sequence design

The initial demonstration uses:

- Proton-PottsMPNN extended vocabulary `v6`
- block descent
- temperature = 0.05
- combined objective λ = 0.3
- HIS-P, ASP-P and GLU-P design centers
- maximum 10 amino-acid substitutions

These are deliberately conservative settings for the first Colab demonstration.

In [12]:
# Run Proton-PottsMPNN pH-sensitive design

RESULT_JSON = "/content/ProtonPottsMPNN/proton_potts_design.json"

env = os.environ.copy()
env["HBPLUS_PATH"] = HBPLUS_PATH

script = """
import json

from biotite.structure.io.pdb import PDBFile
from mpnn.inference_engines.potts_mpnn_ph import (
    PottsMPNNPHEngine,
    PHDesignCriteria,
)

checkpoint = (
    "/content/ProtonPottsMPNN/"
    "checkpoints/potts_v6_afdb_edge_his0.3_acid0.06/"
    "epoch-0125.ckpt"
)

pdb_file = __PDB_FILE__
binder_chain = __BINDER_CHAIN__
result_json = __RESULT_JSON__

engine = PottsMPNNPHEngine(
    checkpoint_path=checkpoint,
    extended_vocab="v6",
    out_directory=None,
    write_fasta=False,
    write_structures=False,
)

atom_array = PDBFile.read(pdb_file).get_structure(model=1)

crit = PHDesignCriteria(
    method="block_descent",
    backend="potts",
    temperature=0.05,
    samples_per_site=1,
    block_size=3,
    combined_lambda=0.3,
    seed_source="native",

    center_types=[
        "HIS-P",
        "ASP-P",
        "GLU-P",
    ],

    dep_map={
        "HIS-P": ["HIS-S"],
        "ASP-P": ["ASP-D"],
        "GLU-P": ["GLU-D"],
    },

    forbidden_tokens=[
        "HIS-A",
        "ASP-A",
        "GLU-A",
        "UNK",
    ],

    placement_by="scan_potts",
    placement_region=["all"],

    repetitive_window_parents=[
        "ARG",
        "LYS",
        "HIS",
        "ASP",
        "GLU",
    ],

    repetitive_window_radius=2,
    repetitive_window_weight=1.0,

    neighbour_k=16,
    max_mutations=10,
    record_trajectory=True,
)

design_set = engine.run_ph_redesign(
    atom_array=atom_array,
    binder_chain=binder_chain,
    criteria_list=[crit],
    seed=0,
)

design = design_set[0]

result = {
    "binder_chain": design.binder_chain,
    "canonical_sequence": design.canonical_sequence,
    "extended_tokens": design.extended_tokens,
    "sequence_length": len(design.canonical_sequence),
}

with open(result_json, "w") as f:
    json.dump(result, f, indent=2)

print("==========================================")
print("DESIGN COMPLETED")
print("==========================================")

print("Binder chain:", design.binder_chain)
print("Sequence length:", len(design.canonical_sequence))

print()
print("Designed sequence:")
print(design.canonical_sequence)

print()
print("Result saved:")
print(result_json)
"""

script = script.replace(
    "__PDB_FILE__",
    repr(str(PDB_FILE))
)

script = script.replace(
    "__BINDER_CHAIN__",
    repr(BINDER_CHAIN)
)

script = script.replace(
    "__RESULT_JSON__",
    repr(RESULT_JSON)
)

result = subprocess.run(
    [PYTHON, "-c", script],
    capture_output=True,
    text=True,
    env=env,
)

print(result.stdout)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "Proton-PottsMPNN design failed."
    )

DESIGN COMPLETED
Binder chain: A
Sequence length: 57

Designed sequence:
MKAHLEILEEVEKKDEAGEDALKVDTEAIKKAEELGDEGLAEAELAAKKLVEAALAE

Result saved:
/content/ProtonPottsMPNN/proton_potts_design.json


STDERR:
Environment variable CCD_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository
Environment variable PDB_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository
/content/ProtonPottsMPNN/.venv/lib/python3.12/site-packages/biotite/structure/io/p

## 9. Generate the design report

The following cell automatically extracts the WT sequence from the uploaded PDB and compares it with the Proton-PottsMPNN sequence.

It reports:

- amino-acid substitutions
- protonation-state tokens
- protonation-state counts
- designed FASTA
- residue-level CSV

In [13]:
# Step 9 — Generate design report and output files

import os
import subprocess
from pathlib import Path

PYTHON = "/content/ProtonPottsMPNN/.venv/bin/python"

RESULT_JSON = Path(
    "/content/ProtonPottsMPNN/proton_potts_design.json"
)

FASTA_FILE = Path(
    "/content/ProtonPottsMPNN/proton_potts_design.fasta"
)

CSV_FILE = Path(
    "/content/ProtonPottsMPNN/proton_potts_mutations.csv"
)

env = os.environ.copy()
env["HBPLUS_PATH"] = HBPLUS_PATH

script = """
import csv
import json

from biotite.structure.io.pdb import PDBFile


PDB_FILE = __PDB_FILE__
RESULT_JSON = __RESULT_JSON__
FASTA_FILE = __FASTA_FILE__
CSV_FILE = __CSV_FILE__

# --------------------------------------------------
# Load Proton-PottsMPNN result
# --------------------------------------------------

with open(RESULT_JSON, "r") as f:
    result = json.load(f)

designed_sequence = result["canonical_sequence"]
extended_tokens = result["extended_tokens"]
binder_chain = result["binder_chain"]

# --------------------------------------------------
# Extract original sequence from PDB
# --------------------------------------------------

atom_array = PDBFile.read(PDB_FILE).get_structure(model=1)

chain = atom_array[
    atom_array.chain_id == binder_chain
]

residue_keys = []

for res_name, res_id in zip(
    chain.res_name,
    chain.res_id
):

    key = (int(res_id), res_name)

    if key not in residue_keys:
        residue_keys.append(key)


AA3_TO_AA1 = {
    "ALA": "A",
    "ARG": "R",
    "ASN": "N",
    "ASP": "D",
    "CYS": "C",
    "GLN": "Q",
    "GLU": "E",
    "GLY": "G",
    "HIS": "H",
    "ILE": "I",
    "LEU": "L",
    "LYS": "K",
    "MET": "M",
    "PHE": "F",
    "PRO": "P",
    "SER": "S",
    "THR": "T",
    "TRP": "W",
    "TYR": "Y",
    "VAL": "V",
}

original_sequence = "".join(
    AA3_TO_AA1[res_name]
    for _, res_name in residue_keys
)

# --------------------------------------------------
# Validate lengths
# --------------------------------------------------

if len(original_sequence) != len(designed_sequence):
    raise ValueError(
        "Original and designed sequences have different lengths: "
        f"{len(original_sequence)} vs {len(designed_sequence)}"
    )

if len(extended_tokens) != len(designed_sequence):
    raise ValueError(
        "Extended-token count does not match designed sequence length."
    )

# --------------------------------------------------
# Mutation table
# --------------------------------------------------

rows = []

for i, (original, designed, token) in enumerate(
    zip(
        original_sequence,
        designed_sequence,
        extended_tokens
    ),
    start=1,
):

    mutation = ""

    if original != designed:
        mutation = f"{original}{i}{designed}"

    protonation_state = ""

    if "-" in token:
        protonation_state = token.rsplit("-", 1)[1]

    rows.append(
        {
            "position": i,
            "original": original,
            "designed": designed,
            "mutation": mutation,
            "extended_token": token,
            "protonation_state": protonation_state,
        }
    )

# --------------------------------------------------
# Write JSON
# --------------------------------------------------

with open(RESULT_JSON, "w") as f:
    json.dump(
        {
            "binder_chain": binder_chain,
            "canonical_sequence": designed_sequence,
            "extended_tokens": extended_tokens,
            "sequence_length": len(designed_sequence),
        },
        f,
        indent=2,
    )

# --------------------------------------------------
# Write FASTA
# --------------------------------------------------

with open(FASTA_FILE, "w") as f:

    f.write(
        f">Proton-PottsMPNN_design_chain_{binder_chain}\\n"
    )

    f.write(
        designed_sequence + "\\n"
    )

# --------------------------------------------------
# Write mutation CSV
# --------------------------------------------------

with open(
    CSV_FILE,
    "w",
    newline=""
) as f:

    fieldnames = [
        "position",
        "original",
        "designed",
        "mutation",
        "extended_token",
        "protonation_state",
    ]

    writer = csv.DictWriter(
        f,
        fieldnames=fieldnames
    )

    writer.writeheader()

    writer.writerows(rows)

# --------------------------------------------------
# Report
# --------------------------------------------------

mutations = [
    row for row in rows
    if row["mutation"]
]

protonation_tokens = [
    row for row in rows
    if row["protonation_state"]
]

from collections import Counter

state_counts = Counter(
    row["protonation_state"]
    for row in protonation_tokens
)

print("==========================================")
print("PROTON-POTTSMPNN DESIGN REPORT")
print("==========================================")

print()
print("PDB:")
print(PDB_FILE)

print()
print("Binder chain:")
print(binder_chain)

print()
print("Sequence length:")
print(len(designed_sequence))

print()
print("Original sequence:")
print(original_sequence)

print()
print("Designed sequence:")
print(designed_sequence)

print()
print("==========================================")
print("AMINO-ACID SUBSTITUTIONS")
print("==========================================")

for row in mutations:

    print(
        f'{row["mutation"]:8s} '
        f'extended_token={row["extended_token"]}'
    )

print()
print(
    "Total amino-acid substitutions:",
    len(mutations)
)

print()
print("==========================================")
print("PROTONATION-STATE TOKENS")
print("==========================================")

for row in protonation_tokens:

    print(
        f'Position {row["position"]}: '
        f'{row["original"]} -> '
        f'{row["extended_token"]}'
    )

print()
print(
    "Total protonation-state tokens:",
    len(protonation_tokens)
)

print()
print("==========================================")
print("PROTONATION-STATE COUNTS")
print("==========================================")

for state, count in sorted(
    state_counts.items()
):
    print(
        f"{state}: {count}"
    )

print()
print("==========================================")
print("FILES CREATED")
print("==========================================")

print()
print("JSON:")
print(RESULT_JSON)

print()
print("Designed FASTA:")
print(FASTA_FILE)

print()
print("Mutation CSV:")
print(CSV_FILE)
"""

script = script.replace(
    "__PDB_FILE__",
    repr(str(PDB_FILE))
)

script = script.replace(
    "__RESULT_JSON__",
    repr(str(RESULT_JSON))
)

script = script.replace(
    "__FASTA_FILE__",
    repr(str(FASTA_FILE))
)

script = script.replace(
    "__CSV_FILE__",
    repr(str(CSV_FILE))
)

result = subprocess.run(
    [PYTHON, "-c", script],
    capture_output=True,
    text=True,
    env=env,
)

print(result.stdout)

if result.stderr:
    print("\nSTDERR:")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        f"Design report generation failed with return code "
        f"{result.returncode}"
    )

PROTON-POTTSMPNN DESIGN REPORT

PDB:
/content/ProtonPottsMPNN/SKB_EGFR_01.pdb

Binder chain:
A

Sequence length:
57

Original sequence:
MKERHEILEEVEKKDEAGEDALKVLEEAIKKAEELGDEGLAETLRAQKKLVEAALAE

Designed sequence:
MKAHLEILEEVEKKDEAGEDALKVDTEAIKKAEELGDEGLAEAELAAKKLVEAALAE

AMINO-ACID SUBSTITUTIONS
E3A      extended_token=ALA
R4H      extended_token=HIS-P
H5L      extended_token=LEU
L25D     extended_token=ASP-P
E26T     extended_token=THR
T43A     extended_token=ALA
L44E     extended_token=GLU-P
R45L     extended_token=LEU
Q47A     extended_token=ALA

Total amino-acid substitutions: 9

PROTONATION-STATE TOKENS
Position 4: R -> HIS-P
Position 6: E -> GLU-D
Position 9: E -> GLU-D
Position 10: E -> GLU-D
Position 12: E -> GLU-D
Position 15: D -> ASP-D
Position 16: E -> GLU-D
Position 19: E -> GLU-D
Position 20: D -> ASP-D
Position 25: L -> ASP-P
Position 27: E -> GLU-D
Position 33: E -> GLU-D
Position 34: E -> GLU-D
Position 37: D -> ASP-D
Position 38: E -> GLU-D
Position 42: E -> GLU-D
Po

In [14]:
# Step 10 — Verify generated output files

from pathlib import Path

OUTPUT_FILES = [
    Path("/content/ProtonPottsMPNN/proton_potts_design.json"),
    Path("/content/ProtonPottsMPNN/proton_potts_design.fasta"),
    Path("/content/ProtonPottsMPNN/proton_potts_mutations.csv"),
]

print("==========================================")
print("OUTPUT FILE VERIFICATION")
print("==========================================")

all_present = True

for file in OUTPUT_FILES:

    exists = file.exists()

    print()
    print(file.name)
    print("  Exists:", exists)

    if exists:
        print(
            "  Size:",
            round(file.stat().st_size / 1024, 2),
            "KB"
        )
    else:
        all_present = False

print()

if not all_present:
    raise FileNotFoundError(
        "One or more expected output files were not created."
    )

print("All expected output files were created successfully.")

OUTPUT FILE VERIFICATION

proton_potts_design.json
  Exists: True
  Size: 0.81 KB

proton_potts_design.fasta
  Exists: True
  Size: 0.09 KB

proton_potts_mutations.csv
  Exists: True
  Size: 0.93 KB

All expected output files were created successfully.


## 10. Interpretation and limitations

A successful Proton-PottsMPNN run demonstrates that the computational design workflow completed successfully.

It does not establish that the resulting protein:

- has the desired pKa values;
- undergoes a useful conformational transition;
- retains its fold;
- retains target binding;
- exhibits the intended pH selectivity;
- is experimentally stable.

For an actual targeted pH-responsive binder, the designed sequence should subsequently undergo structural prediction/relaxation, pKa analysis and experimental validation.

